![hslu_logo.png](./img/hslu_logo.png)


<hr style="border:1px solid black">

<h1 style="text-align:center;font-size:50px"><b>AAI - FS25</b></h1>
<p style="text-align:center;font-size:40px">Week 03</p>

---

# Convolutional Neural Networks (NN)

---
---

 
# Table of contents for week 02
1. [Convolutional Neural Networks](#conv_nn)
     1. [Visual Object Classification with MLP](#class_mlp)
     2. [Convolutions in Image Processing](#conv_img_proc)
     3. [General Architecture of CNNS](#gen_arch_cnn)
        1. [Convolutional Layer](#conv_layer)
        2. [Pooling Layer](#pool_layer)
        3. [Overall CNN Architecture](#over_arch)


# Convolutional Neural Networks <a name="conv_nn"></a>

Convolutional Neural Networks (CNNs) emerged from the study of the visual cortex of mammals, which was found to be organised in hierarchical structures inspiring the CNN architecture.

<br>
<img src="img/visual_cortex.png" alt="Drawing" width="600" />
<a id="fig1">Fig.1:</a> The visual cortex of a human is organized in hierarchical structures. This bears similarities to CNNs.
<br>

---

In a famous study <a id="anker1" href="#ref1">[1]</a> on the visual cortex of cats it was shown that many neurons in the visual cortex have only a small receptive field – partially overlapping – and that different neurons react to different stimuli e.g. to horizontal or vertical lines only. Moreover, it could be shown that certain neurons with larger fields of view react to more complex patterns that are combinations of the “low-level” neuron features. These studies inspired the so-called neocognitron <a id="anker2" href="#ref2">[2]</a>, which gradually evolved in what we now call CNN. An important further milestone was the publication of LeNet-5 by Yann LeCun at al. 
<a id="anker3" href="#ref3">[3]</a> with application of CNN to the recognition of handwritten digits.

Nowadays CNNs are mainly used for visual classification and detection tasks but may be applied to any kind of 2D – or even 3D – data type. In this chapter we will focus on the visual object classification e.g. like the recognition of the handwritten MNIST digits. Visual object detection – i.e. the localisation of visual object categories in images – is also possible with CNNs by applying certain changes or extensions to their architecture. This topic will be treated in the chapter [sw05.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW05/sw05.ipynb#).

CNNs introduce two new layer types which are convolutional and pooling layer. The convolutional layer realises the concept of a neuron with limited receptive field and in addition gives CNN the ability of translational invariance in the detection task. The pooling layer, which is a spatial decimation, introduces the concept of feature hierarchy.

Before we introduce the general architecture of the CNN, we will first recall some concepts of the MLP, which will allow us to understand its limitations in terms of the visual classification task and then have a quick look at the convolution-concept, as it is used in classical image processing.



## Visual Object Classification with MLP <a name="class_mlp"></a>

While computers play chess at human level since the 1997, image recognition at human level has only been possible since about a decade ago. The problem is that a given visual object category may be subject to considerable interclass variation, viewpoint changes, deformations, occlusions, ... <a href="#fig2">Fig.2</a>. A classification task should be able to cope with all these changes. However, the MLP architecture has certain weaknesses that limit their performance on visual classification tasks.

<br>
<img src="img/class_dog.png" alt="Drawing" width="600" />
<a id="fig2">Fig.2:</a> The category «dog» may be subject to considerable interclass variation, viewpoint changes, deformations, occlusions. A classification or detection task should be able to cope with all these changes (Selection of images from the <a href=https://en.wikipedia.org/wiki/CIFAR-10>CIFAR-10</a> image set).
<br>

---


**Exercise:** 
**[sw03.01.mlp_fashion-mnist.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW03/sw03.01.mlp_fashion-mnist.ipynb)**


The script is mainly the solution to the iPython notebook [sw02.04.mlp_fashion-mnist_SGD.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW02/sw02.04.mlp_fashion-mnist_SGD.ipynb#)  from last week with the extension to use a validation data set. Here we want to recall, that the weights of the first layer learn in some sense the structure of the corresponding categories.

- Quickly review the structure of the notebook, as we introduced some minor changes:
    - Cell [10] no contains the split in training and validation data using the validation fraction `validation_size`.<br>
    Furthermore the structure of the MLP is given as list of neurons `[num_input, num_hidden, num_output]` and therefore you can also add more hidden layers. Check the corrsponding implementation in the constructor of the class `MultiLayerPerceptron` 
    - First change the architecture to work without hidden layer and perform a training run by executing Cell [10]. Rather choose a small number of epochs (e.g. 5). 
    - Then move to Cells [13], reshape the weigths of the first (which is also the last) layer to 10 tiles of size 28x28 and plot them. The result should look similar to (<a href="#fig3">Fig.3</a>). <br>

<img src="img/mlp_mnist_0.png" alt="Drawing" width="700" />
<a id="fig3">Fig.3:</a> Weights of a Generalised Perceptron learn the structure of the corresponding object categories.
<br>

---


- Now move back to cell [10] and configure a MLP with one single hidden layer and some 100 neurons:

- Again trigger the training of some 5 epochs and visualise the result of the weights of the first layer. Now it should look similar to <a href="#fig4">Fig.4</a>.

<img src="img/mlp_mnist_1.png" alt="Drawing" width="1000" />
<a id="fig4">Fig.4:</a> Weight vectors of the first hidden layer of a MLP for classification of fashionMNIST data.
<br>

---

From <a href="#fig3">Fig.3</a> we conclude that a Generalised Perceptron just learns some sort of average of all representations of a given category. For the more complex case of a MLP in <a href="#fig4">Fig.4</a> we see that the first hidden layer apparently learns various representations of a given category. Considering the different possible representations of a given category as shown in <a href="#fig2">Fig.2</a>, two obvious problems arise:

1.	The MLP learns the position of an object “by heart” or in other word each position of an object – in particular if the image is larger – would have to be learnt. The algorithm is not intrinsically translation invariant.

2.	So far, we only used small image sizes of some 1000 pixels in total. Consider now the case of a state-of-the-art colour image of 400 x 400 pixel, which leads to 480’000 input values. An MLP with one hidden layer of “only” 1000 neurons would already require 480 million parameters, which would become quickly intractable. 

We will see in the following, how the idea of convolutional layers will allow to solve these two problems i.e., to introduce translational invariance and to considerably reduce the number of parameters. We will in a first step discuss the convolution from the point of view of classical image processing.

## Convolutions in Image Processing <a name="conv_img_proc"></a>

Convolutions are a well-known tool both in digital signal and in image processing. The two most important types are low-pass and high-pass filters. We will focus here on the latter type as these are most relevant for understanding convolutional layers.

<img src="img/conv_01.png" alt="Drawing" width="650" />
<a id="fig5">Fig.5:</a> Convolution of an image $a_{i,j}$ with a kernel (or mask) $w_{m,n}$ (of size 5 x 5) to obtain the result $a'_{i,j}$

---

<a href="#fig5">Fig.5</a> represents the convolution of an image  $a_{i,j}$ with a kernel (or mask) $w_{m,n}$ (of size 5 x 5) to obtain the result $a'_{i,j}$. Therefore, all image pixel in the “receptive” region of the kernel (5 x 5) centred at image pixel $a_{i,j}$ are multiplied with the respective kernel values $w_{m,n}$ and their sum is assigned to $a'_{i,j}$. One can visualise this operation as sliding the kernel – e.g. row-wise – over the full image $a_{i,j}$ to obtain $a'_{i,j}$. Mathematically, the operation for the convolution can be formulated as follows:

<img src="img/conv_def.png" alt="Drawing" width="600" />

However, this immediately raises the question of the treatment of the border pixel. As shown in the following <a href="#fig6">Fig.6</a>, if we want to preserve the size of the original image $a_{i,j}$ – and e.g. as shown in the figure want to treat pixel $a_{0,0}$ – we have to extend (“pad”) the original image to provide the missing values to the kernel. Default padding is usually to extend the original image with zeros, but more robust versions exist like reflecting the image pixels at the border to avoid artefacts arising from discontinuities due to zero-padding.

<img src="img/conv_02.png" alt="Drawing" width="700" />
<a id="fig6">Fig.6:</a> To preserve the size of the original image $a_{i,j}$ padding is required at the borders.
<br>

---

For the application of convolutions in image processing typical kernel choices of size 3 x 3 for edge detection (high-pass filter) in x- or y-direction are (left and right respectively):

<img src="img/conv_dx_dy.png" alt="Drawing" width="700" />

Note the sign-convention for the y-direction (minus-sign in top line of kernel) which is because the y-index increases from top to bottom.

While edge detection filters represent an approximation for the first derivative the second derivative can be approximated with the Laplace filter:

<img src="img/conv_lapl.png" alt="Drawing" width="500" />

The classic formulation of convolutions as summarised above usually deals only with single channel input images. This is the case illustrated in <a href="#fig7">Fig.7</a> on the left-hand side, where a kernel $w_{m,n}$ is applied on a MNIST or FashionMNIST image $a_{i,j}$ of size 28 x 28 pixel. 

<img src="img/conv_03.png" alt="Drawing" width="600" />
<a id="fig7">Fig.7</a> (left) Classical convolution on a single channel image and extension (right) to three channel case (details c.f. text). Also note the 4D-indices $(b,c,i,j)$ of the image including $b$ the index of the image in the batch.
<br>

---

For use in CNNs we will extend the definition of convolutions to multi-channel input images. This is represented for three channel i.e., colour images in <a href="#fig7">Fig.7</a> on the right. Input is a CIFAR-10 image with three colour channels $a_{i,j}^c$ i.e., $c=0,1,2$. Now the kernel also must have three channels $w_{c,m,n}$ where $c=0,1,2$. Thus, the kernel now has three channels e.g. of size 3 x 3 (other types possible and frequent) and the values of the three channels can obviously be different. The formula for the convolution now reads:

<img src="img/conv_def_01.png" alt="Drawing" width="600" />


**Exercise:** 
**[sw03.02.intro_conv.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW03/sw03.02.intro_conv.ipynb)**

We use this iPython notebook to review the concept of convolutions in image processing for single channel images and extend it to multi-channel:

- Quickly review the structure of the notebook:
    - Cell [2] slightly changed because we require the images now to include the dimension of the colour plane, because we will start to work with the CIFAR-10 colour images. For reason of compatibility the single channel images (MNIST and fashionMNIST) also must provide this additional dimension. <br>
    Therefore we use the new function `read_data_orig` (from `utils.py`) which provides the data (with an additional `unsqueeze`) as 4D-array.   
    - You may first want to start with the single channel images (MNIST or fashionMNIST). If you read them you will notice the new array dimensions which are `torch.Size([60000, 1, 28, 28])`
    - Then you may try `CIFAR10` and observe the array dimension which are `torch.Size([50000, 3, 32, 32])`. But for a first application of the convolution move back to  MNIST or fashionMNIST data.
    - Cells [3] - [5] illustrate how to plot some sample images. As the colour plane is at the first position ("channels-first") it may be necessary to switch the channel order for certain plot functions (as e.g. matplotlib). This is the reason for the call to `torch.movedim`.
    - Cell [6] now allows to apply the convolution. At the top you can choose the object category and further below the kernel type. The edge detection in x-direction is implemented. You have to complete the filters for the edge detection in y-direction and the Laplace filter.

<img src="img/exec_02_01.png" alt="Drawing" width="800" /> 
  
- Execute the cell [6] and observe the output. You wil notice that the output size changed because the padding is not activated.

<img src="img/exec_02_02.png" alt="Drawing" width="700" /> 
  
- The script uses the torch functional API to realise the convolution. Look-up the documentation for the function [conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html#torch.nn.Conv2d) and implement the padding such that the size of the image is conserved. Verify with the output shape printed in the console.

<img src="img/exec_02_03.png" alt="Drawing" width="700" />
  
- Move up to cell [2] again, choose the `CIFAR10` image data and investigate the results. In particular verify the size of the in- and output with respect to the channel size (red  arrows).

<img src="img/exec_02_04.png" alt="Drawing" width="300" />

- Implement the two missing filters and verify the correctness of the result. E.g. the Laplace filter would look as shown in <a href="#fig8">Fig.8</a>. Note also the artefacts that appear due to the zero padding of the function `torchvision.utils.make_grid(..)` used to create the tile-images:

<img src="img/exec_02_05.png" alt="Drawing" width="650" />

<br>
<img src="img/conv_cifar_lapl.png" alt="Drawing" width="700" />
<a id="fig8">Fig.8:</a> Laplace-filter as an example for a convolution on a multi-channel image (CIFAR-10).
<br>

---


## General Architecture of CNNS <a name="gen_arch_cnn"></a>

The key elements of the CNN are the convolutional and the pooling layer, which were inspired by the study on the visual cortex of cats <a id="anker1" href="#ref1">[1]</a>. In fact, the main building block of CNNs is a set of convolutional layers (each including a non-linear activation function) followed by a pooling layer. This block is then repeated several times until the spatial resolution is small enough to flatten the result to a single vector and to apply a standard MLP with dense layers as classifier.<br>
We will start with the convolutional layer and treat the pooling layer in the next section <a href="#pool_layer">Pooling Layer</a>. 


### Convolutional Layer <a name="conv_layer"></a>

The illustration of the convolution in <a href="#fig5">Fig.5</a> (c.f. also <a href="#fig12">Fig.12</a>) shows that each output neuron – here positioned at $a'_{i,j}$ – has only a limited receptive field determined by the size of the kernel $w_{m,n}$. In addition, the same kernel is used at all positions or in other words, it is shared between all output neurons. Therefore, it requires much less parameters as compared to a dense layer of comparable size. Furthermore, the features – e.g. as shown in <a href="#fig8">Fig.8</a> – are detected everywhere in the image. Thus, the detection is intrinsically translation invariant. Therefore, convolutions allow to solve the two problems – stated at the end of section <a href="#class_mlp">Visual Object Classification with MLP</a> – related to object classification using MLP with dense layers. However, it is immediately clear that only very simple features – like edges in <a href="#fig8">Fig.8</a> – could be determined with a kernel of limited receptive field. This is where the pooling layer comes into play to reduce iteratively the size of the image which – in turn – increases the complexity of the detect features. This will be discussed in detail in the next section <a href="#pool_layer">Pooling Layer</a>.<br>
So far, we only applied a single kernel to the input images. However, a single kernel allows to detect only one typical feature type – e.g. as shown in <a href="#fig8">Fig.8</a> – but for the classification of a large number of object categories we will certainly require many different features. Therefore, we will extend the single kernel case shown in <a href="#fig7">Fig.7</a> to multiple kernels. This is illustrated in the following <a href="#fig9">Fig.9</a>. 

<img src="img/conv_mnist.png" alt="Drawing" width="450" />
<a id="fig9">Fig.9:</a> Extension of the convolution to a set of 4 kernels $c'=4$.
<br>

---

As shown the input to the convolution operation is a multi-channel image $a_{i,j}^c$ with a total of $c=6$. We consider images of size 28 x 28 (i x j). The convolution now comprises a total of $c'=4$ kernels and we have to add an additional index to the kernel elements $w_{c,m,n}^{c'}$. Application of each of the $c'=4$ individual kernels will give as output a single channel image with the same output size (we assume padding). Thus, the output “image” – or activation map – $a_{i,j}^{c'}$ will have a total of $c'=4$ channels. <br>
The formula for the convolution now reads :

<img src="img/conv_def_02.png" alt="Drawing" width="700" />

Note that we added the bias $b^{c'}$, which – for each value of $c'$ – is a single scalar value. Note further, that the convolution is applied to a batch consisting of $b$ images. In <a href="#fig10">Fig.10</a> we extended the above illustration to represent the case of $b=5$
.

<img src="img/conv_mnist_batch.png" alt="Drawing" width="600" />
<a id="fig10">Fig.10:</a> Application of a convolution (input channels $c=6$, output channels $c'=4$) to a batch of $b=5$ images.
<br>

---

So far, we limited the convolution operation to shift the kernel in each step – row or column – by one pixel. This is the so-called stride. It is also possible to extend the convolution operation to larger strides. Obviously, this will reduce the output size of the image. In <a href="#fig11">Fig.11</a> this is illustrated for the case of a 
10 x 10 input image and a 3 x 3 kernel size. If a “standard” stride $S=1$ (top) is applied, the kernel will be applied to all pixel of the input image $a_{i,j}$ and the output $a'_{i,j}$ will have the same size (i.e. 10 x 10). If we apply a stride of $S=2$ (bottom) the kernel will only be applied to every other pixel (grey) row- and column-wise. Therefore, the output will have a reduce size of 5 x 5 for the illustrated numbers.

<img src="img/conv_stride.png" alt="Drawing" width="550" />
<a id="fig11">Fig.11:</a> Application of different strides ($S=1$ top, $S=2$ bottom) leads to different output sizes (padding assumed).
<br>

---

In general, the output size can be evaluated depending on the different parameters according to the following formulas:

<img src="img/stride_padd.png" alt="Drawing" width="700" />

We apply integer division i.e., rounding to the lower value. Furthermore, the meaning of the variables are as follows:

- $i$,$i'$,$j$,$j'$: number of rows and columns of in- and output images, respectively.
- $m$,$n$: number of rows and columns of the kernel.
- $p_i$,$p_j$: number of pixels padded in row and column direction, respectively.
- $S_i$,$S_j$: stride in row and column direction, respectively.

For the example given in <a href="#fig11">Fig.11</a> application of the formula (row and column direction is identical) for the $S=2$ case leads to:

<img src="img/stride_padd_01.png" alt="Drawing" width="500" />

#### Relation to dense Layer

It is instructive to consider the relationship between the convolutional layer and the dense layer. Therefore, we consider a simple convolutional layer with only one input and output channel as represented in <a href="#fig12">Fig.12</a>. The kernel has the size 3 x 3 and we do not apply padding. In addition, we made the input and output neurons – i.e. the pixel – explicit. The input channel has 14 x 11 pixel (neurons) and – due to the absence of padding – the output channel only 12 x 9. Three corresponding in- and output neurons are highlighted with the corresponding kernel position. We see on the one hand the limited receptive field of the output neurons within the input image and that on the other that the spatial relationship between the in- and output neurons is preserved.



<img src="img/conv_3x3.png" alt="Drawing" width="400" />
<a id="fig12">Fig.12:</a> A simple convolutional layer with one input and output channel (details c.f. text).
<br>

---

We now increase the kernel size to – first – 5 x 5 and then to 14 x 11 i.e. to the size of the input channel (left and right in <a href="#fig13">Fig.13</a> respectively). For kernel size 5 x 5 the output channel is already reduced to a size of 10 x 7. If we finally increase the kernel size to 14 x 11 the output channel will consist of only one single pixel (neuron) and – in addition – the spatial information will be completely lost because this output neurons receives the information of all input pixel. 

<img src="img/conv_5x5.png" alt="Drawing" width="750" />
<a id="fig13">Fig.13:</a> Increasing the size of the kernel will – without padding – decrease the size of the output activation (details c.f. text).
<br>

---

Because the spatial information is lost, there is no further need to represent the input pixel as a 2D-image array, but we can flatten them to a single input vector (<a href="#fig14">Fig.14</a>, right). This will give a total of 1 x 14 x 11 = 154 input pixel (neurons) with a single output neuron. Thus, we see that a convolutional layer with a kernel size equal to the input image – and without padding – can be considered as a dense layer with one output neuron. Additional output neurons would then correspond to additional kernels of the same size i.e. 1 x 154.

<img src="img/conv_mxn.png" alt="Drawing" width="650" />
<a id="fig14">Fig.14:</a> A «convolutional layer» with kernel size corresponding to the input image size (left) is equivalent to a dense layer with a single output neuron.
<br>

---

Thus, e.g. application of a dense layer with four output neurons on a (Fashion)MNIST image represented as convolutional layer would look like illustrated in the following <a href="#fig15">Fig.15</a>. 

<img src="img/conv_dense_conv.png" alt="Drawing" width="500" />
<a id="fig15">Fig.15:</a> Dense layer with four output neurons represented as convolutional layer.
<br>

---

### Pooling Layer <a name="pool_layer"></a>

In the previous section we illustrated (e.g. <a href="#fig12">Fig.12</a>) how the convolutional layer preserves the spatial information from the input image. However, the limited receptive field would allow to detect only relatively low-level features like edges, corners, etc. This is where the pooling layer comes into play to reduce iteratively the size of the image which – in turn – increases the complexity of the detect features. The idea is conceptually simple and is illustrated in the following <a href="#fig16">Fig.16</a>. 

<img src="img/pooling.png" alt="Drawing" width="500" />
<a id="fig16">Fig.16:</a> Pooling applied to a 4 x 6-pixel image patch. Max pooling top, average pooling bottom (average values rounded to 2 significant digits).
<br>

---

The choice of parameters represented is the most frequent one: a pooling size of $P=2$ with a stride of $S=2$. This means that the pooling is performed over each sub-patch of size 2x2  and the patch is always shifted by 2 pixel in row and column direction. For the pooling the most frequent option is max pooling, where the maximum value of the activations in the sub-patch is chosen (top in <a href="#fig16">Fig.16</a>). In addition, average pooling is a frequent option, where the average value of the activations in the patch is calculated (bottom in <a href="#fig16">Fig.16</a>). <br>
The idea of max pooling is that the most prominent activation in the pooling region is selected and then passed to the next layer, which – as we will see in the next section – is usually a convolutional layer. Due to the image decimation the receptive field of the convolutional layer will now cover a larger part (twice as much for the settings in <a href="#fig16">Fig.16</a>) of the original image and can therefore detect features of higher complexity. This alternating sequence of convolutional and pooling layers is the main architectural paradigm of CNNs. It is continued to a point, where the feature complexity is high enough such that a “standard” MLP classifier can be used for the object classification.

### Overall CNN Architecture <a name="over_arch"></a>

As already mentioned above the main characteristics of CNN is the alternating sequence of convolutional and pooling layers followed – after a flattening – by one or two dense layers for the actual classification. In the following we will - as an example - implement the architecture represented in <a href="#fig17">Fig.17</a>. The representation is for (Fashion)MNIST images i.e., input image size 1 x 28 x 28 but can be easily adjusted for CIFAR-10 images with size 3 x 32 x 32.<br>
The first layer is a convolutional layer with 16 kernels of size 5 x 5 applied at stride $S=1$. We will always apply padding to preserve the original image size. The result will be an activation map with 16 channels i.e., 16 x 28 x28. Then we apply a max pooing layer of pooling size $P=2$ with stride $S=2$. We obtain output activations of size 16 x 14 x 14. Now we apply a second convolutional layer with 32 kernels of size 3 x 3 to obtain an output of size 32 x 14 x 14, again followed by a max pooling layer ($P=2$, $S=2$.) to obtain the intermediate result of size 32 x 7 x 7. <br>
This is the point where we consider the features to be of sufficient complexity to apply a standard MLP classifier. Therefore, the output is flattened – to a vector of 32 x 7 x 7 = 1568 element – which is the input to the first dense layer with 200 neurons followed by the final softmax output with the 10 classes.
We will see below that PyTorch offers convenient functionality to implement and parametrise theses layers in a few lines of code.

<img src="img/cnn_final.png" alt="Drawing" width="900" />
<a id="fig17">Fig.17:</a> CNN-architecture that will implemented in PyTorch and TensorFlow below (details c.f. text).
<br>

---

Several important improvements and/or extensions to this standard architecture have been proposed since the publication of Alexnet <a id="anker4" href="#ref4">[4]</a>. Here we will only mention VGG16 <a id="anker5" href="#ref5">[5]</a> because its architecture extends the above scheme with a very common idea consisting in a stack of several consecutive convolutional layers before applying a pooling layer.

The following <a href="#fig18">Fig.18</a> gives an overview of the VGG16 architecture, which consists of stacks of 2 or 3 consecutive convolutional layers followed by a pooling layer. The detailed layer sizes are summarised in <a href="#tab19">Tab.19</a>. There, in addition the number of parameters per layer – only weights no biases – are given. Observe the ratio of parameters in the convolutional with respect to the dense layers. This is a consequence of the weight sharing of the neurons in the convolutional layer (<a href="#fig12">Fig.12</a>).

<img src="img/vgg16.png" alt="Drawing" width="600" />
<a id="fig18">Fig.18:</a> Architecture of the VGG16 CNN <a id="anker5" href="#ref5">[5]</a>.
<br>

---

<img src="img/vgg16_layer.png" alt="Drawing" width="600" />
<a id="tab19">Tab.19:</a> Layer sizes for the VGG16 CNN architecture and corresponding number of parameters (only weights no biases).
<br>

---


**Exercise:** 
**[sw03.03.cnn.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW03/sw03.03.cnn.ipynb)**

We use this iPython notebook to implement a CNN for image classification using single or multiple channel image input. 

- Quickly review the structure at the beginning of the notebook:
    - Cell [2] is identical to [sw03.02.intro_conv.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW03/sw03.02.intro_conv.ipynb) with the choice of (Fashion)MNIST or CIFAR-10 data.      
    - Cells [3] - [5] are also identical to [sw03.02.intro_conv.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW03/sw03.02.intro_conv.ipynb) with output of some sample images.
    - Cell [6] is our own dataloader that provides the image batches during the training.<br>
- Move on to cell [7] `Define the neural network`<br>
    Here you are supposed to implement the missing code parts 

  *### START YOUR CODE ###*

   *### END YOUR CODE ###*

  Before you start with the programming have a look at cell [9] (`Define X and Y values and do optimization`), where the network architecture is configured:<br>
  `cnn = ConvNeuralNetwork(size_in, list_conv_layer, list_num_neurons)`

  The code provided in [sw03.01.mlp_fashion-mnist.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW03/sw03.01.mlp_fashion-mnist.ipynb) gives an example on how to use the list of hidden neurons (`list_num_neurons`) to configure the corresponding number of dense layers in the constructor of the cnn class. In addition, you now have the list of parameters for the convolutional layers available (`list_conv_layer`) that allows to configure the corresponding number of layers. This represents the main task to peform.<br>
  The present set `list_conv_layer = [[16,5],[32,3]]` corresponds to the settings of <a href="#fig17">Fig.17</a> with two convolutional layers having, respectively, 16 and 32 kernels. Furthermore, the first set of kernels is of size 5 x 5 and the second of size 3 x 3.<br>
  In addition note the parameter `size_in`, which in the present case is `[1, 28, 28]`. You will require this information to configure the first convolutional layer in the cnn class.
  
  You may now proceed to the implementation of the cnn in cell [9]. Refer to the Pytorch [documentation](https://pytorch.org/docs/stable/nn.html) for the details of the different layers. In particular, the following layer types will be of interest:
  - torch.nn.Conv2d(..)
    You have to specify the parameters for the
    - in_channels: number of channels of input (size_in) or previous layer
    - out_channels: number of kernels in layer e.g. 16 for the first conv layer
    - kernel_size: e.g. 5 (corresponding to 5 x 5) for the first conv layer
    - In addition make sure to set padding correctly, to preserve the size:<br>
      padding='same'<br>
      It is also advisable to use a padding option with a minimum of artefacts at the borders e.g. reflect:<br>
      padding_mode='reflect'    
  - torch.nn.ReLU()
      We already know the ReLU activation function from the previous exercises.
  - torch.nn.MaxPool2d(..)
    You have to specify the pooling size:
    - kernel_size: We always choose a value of $P=2$
    - The stride is per default identical to the kernel_size so can leave the default (“None”).
  - torch.nn.flatten(..)
    Before the first dense layer the output of the last convolutional layer has to be flattend. This is done with a flatten layer.
 - In addition you will have to implement the dense layers – including their activations. Note that you have to specify the size of the input for the first dense layer, which depends on the detailed parameters of the convolutional layers. For our choice (<a href="#fig17">Fig.17</a>) the number would be 32 x 7 x 7 = 1568.
 - Implement the full CNN and trigger the trainig using cell [9]. Depeding on the CPU power the training might take several minutes.
 - After the training Execute cell [10] (`Analyse the number of parameters`) to visualise the parameters of the different layers. Observe the ratio of parameters between the convolutional and dense layers and review <a href="#tab19">Tab.19</a> with the VGG16 architecture and parameter counts.
 - Then move on to the cells [11] and [12], where we want to analyse the result of the convolutional layers.
   - Cell [11] plots the weights of the first convolutional layer:<br>
   `cnn.model[0].weight`<br>
   Make sure to understand the above syntax. The weights – shown as tile image – may look as follows (each training will be different):
<img src="img/conv_weights.png" alt="Drawing" width="400"/>

    - Cell [12] allows to visualise the activation maps.<br>
      For the first convolutional layer this reads:<br>
      `activ=CNN.model[0](img)`<br>
      This represents the application of the convolution only i.e. without activation function and the output are the 16 activation maps from the first layer, with shape [1, 16, 28, 28]. Make sure to understand the implementation and verify the output shape. An example for the original image and the activation maps are given below:
      <img src="img/conv_activ.png" alt="Drawing" width="600"/>
      
      For the second convolutional layer we use:<br>
      `for i0 in range(1,4):`<br>
          `activ=CNN.model[i0](activ)`<br>
      This loop applies successively layers 1 to 3 (activation function of 1st conv layer, 1st pooling layer, 2nd conv layer) and will therefore give the activation maps of the 2nd conv layer. Verify that their shape is indeed [1, 32, 14, 14]. Make sure to understand how the size is determined by the different layers.<br>
      Below these activation maps are plotted for the input image shown above:
      <img src="img/conv_activ_02.png" alt="Drawing" width="400" />

#### Use of Existing Model Architectures

In practice it is now rather uncommon that we train a neural network from scratch. The reason is that the deep learning framework PyTorch (and others as TensorFlow as well) provide a large class of pretrained models (“model zoo”) that can be downloaded and used. 
In the following exercise we will illustrate how to download and apply the vgg16 network (<a href="#fig18">Fig.18</a>), which yet has a rather simple architecture. 
One further important aspect is the so-called fine-tuning of existing model architectures. Therefore, a given model architecture is retrained with own data usually of limited size. The idea behind is to use the low-level features (convolutional layers) of the pretrained model and only fine tune the classifier (dense layer) with a limited set of own data. This works often amazingly well. We will deal with this important topic in the next chapter [sw04.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW04/sw04.ipynb#).

**Exercise:** 
**[sw03.04.model_zoo.ipynb](http://localhost:8888/notebooks/Kursunterlagen/SW03/sw03.04.model_zoo.ipynb)**

We use this iPython notebook to illustrate the use of a pretrained vgg16 network architecture. Extensions to other networks are straight foreward.

The notebook is rather self-explaining and you should be able to work yourself through the notebook with the comment given. Do some experiments with other images, which you can read as shown in cell [7]. Furhtermore you may want to explore other - more complex - network architectures form the `model zoo` listed in cell [3].

---
---

<a id="ref1" href="#anker1">[1]</a>  Hubel, David H., and Torsten N. Wiesel. "Receptive fields, binocular interaction and func-tional architecture in the cat's visual cortex." The Journal of physiology 160.1 (1962): 106.

<a id="ref2" href="#anker2">[2]</a> Fukushima, Kunihiko. "Neocognitron: A hierarchical neural network capable of visual pattern recognition." Neural networks 1.2 (1988): 119-130.

<a id="ref3" href="#anker3">[3]</a> LeCun, Yann, et al. "Gradient-based learning applied to document recognition." Proceedings of the IEEE 86.11 (1998): 2278-2324.

<a id="ref4" href="#anker4">[4]</a> Krizhevsky, Alex, Ilya Sutskever, and Geoffrey E. Hinton. "Imagenet classification with deep convolutional neural networks." Advances in neural information processing systems 25 (2012).

<a id="ref5" href="#anker5">[5]</a> Simonyan, Karen, and Andrew Zisserman. "Very deep convolutional networks for large-scale image recognition." arXiv preprint arXiv:1409.1556 (2014).